In [1]:
import os
import json
import random
import re
import yaml
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Allowed language codes for your experiments
LANGUAGE_CODES = [
    "bg", "zh", "nl", "en", "fr", "de", "id", "fa", "uk", "af", "ar", "eu", "et",
    "el", "he", "it", "ja", "pl", "pt", "sr", "es", "sv", "cy", "yue", "ace", "ban",
    "bug", "hr", "cs", "da", "hu", "is", "jv", "ko", "mak", "min", "no", "nso", "ro",
    "ru", "st", "su", "tr", "xh", "zu"
]

ISO_639_1_TO_3 = {
    "bg": "bul", "zh": "zho", "nl": "nld", "en": "eng", "fr": "fra", "de": "deu",
    "id": "ind", "fa": "pes", "uk": "ukr", "af": "afr", "ar": "arb", "eu": "eus",
    "et": "est", "el": "ell", "he": "heb", "it": "ita", "ja": "jpn", "pl": "pol",
    "pt": "por", "sr": "srp", "es": "spa", "sv": "swe", "cy": "cym", "yue": "yue",
    "ace": "ace", "ban": "ban", "bug": "bug", "hr": "hrv", "cs": "ces", "da": "dan",
    "hu": "hun", "is": "isl", "jv": "jav", "ko": "kor", "mak": "mak", "min": "min",
    "no": "nob", "nso": "nso", "ro": "ron", "ru": "rus", "st": "sot", "su": "sun",
    "tr": "tur", "xh": "xho", "zu": "zul"
}

FULL_LANGUAGE_NAMES = {
    "sq": "Albanian",
    "bg": "Bulgarian",
    "ms": "Malay",
    "nl": "Dutch",
    "ko": "Korean",
    "mk": "North Macedonian",
    "sr": "Serbian",
    "ml": "Malayalam",
    "vi": "Vietnamese",
    "hi": "Hindi",
    "ja": "Japanese",
    "de": "German",
    "et": "Estonian",
    "lt": "Lithuanian",
    "hu": "Hungarian",
    "ur": "Urdu",
    "uz": "Uzbek",
    "te": "Telugu",
    "kk": "Kazakh",
    "tr": "Turkish",
    "ru": "Russian",
    "ka": "Georgian",
    "it": "Italian",
    "tl": "Tagalog",
    "el": "Greek",
    "pt": "Portuguese",
    "he": "Hebrew",
    "fi": "Finnish",
    "pl": "Polish",
    "hr": "Croatian",
    "fr": "French",
    "be": "Belarusian",
    "hy": "Armenian",
    "az": "Azerbaijani",
    "es": "Spanish",
    "ta": "Tamil",
    "zh": "Chinese",
    "fa": "Persian",
    "id": "Indonesian",
    "ar": "Arabic",
    "ne": "Nepali",
    "uk": "Ukrainian",
    "bn": "Bengali",
    "eu": "Basque"
}

DATASET_CLEANING_REGEX = {
    "truthfulqa": re.compile(
        r"Question:\s*|Answer:\s*|Answer\s+with\s+A,\s*B,\s*C,\s*D,\s*\.\.\.",
        re.IGNORECASE
    ),
    "mmlu": re.compile(
    r"Question:\s*|Choice\s|Answer with\s*(?:,\s*)*.*?\nAnswer:",
    re.IGNORECASE | re.DOTALL
),
    "arc": re.compile(
        r"Question:\s*|Answer with\s*[A-D](?:,\s*[A-D])*.*?\nAnswer:",
        re.IGNORECASE | re.DOTALL
    ),
    "bmlama": re.compile(
        r"Question:\s*|Answer with\s*[A-D](?:,\s*[A-D])*.*?\nAnswer:",
        re.IGNORECASE | re.DOTALL
    ),

    "default": re.compile(
        r"Answer\s+with\s+A,\s*B,\s*C,\s*D,\s*\.\.\.", re.IGNORECASE
    ),
}


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_lang_mapping(yaml_path: str, language_codes: list) -> dict:
    """
    Parses a YAML file to map ISO 639-1 language codes to ISO 639-3 + script codes.
    
    Args:
        yaml_path (str): Path to the YAML file.
        language_codes (list): List of ISO 639-1 codes to include in the mapping.
    
    Returns:
        dict: Mapping of ISO 639-1 code → ISO 639-3_script code.
    """

    
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    tasks = data.get("task", [])
    lang_mapping = {}

    for entry in tasks:
        if not entry.startswith("sib200_"):
            continue

        code = entry[len("sib200_"):]  # e.g. "bul_Cyrl"
        parts = code.split("_")
        if len(parts) != 2:
            continue

        lang_iso3, script_code = parts

        # Reverse lookup to ISO-639-1
        for iso1, iso3 in ISO_639_1_TO_3.items():
            if iso3 == lang_iso3 and iso1 in language_codes:
                lang_mapping[iso1] = f"{iso3}_{script_code}"
                break  # Found match, no need to check others

    return lang_mapping



def load_and_sample_dataset(dataset_name: str, language_1: str, max_examples: int = 10000):
    """
    Loads HuggingFace dataset for the given language_1.
    Optionally subsamples to max_examples.
    """

    if dataset_name in ["pkavumba/balanced-copa", "HiTZ/XCOPA-eu"]:
        try:
            dataset = load_dataset(dataset_name)
        
        except ValueError as e:
            print(f"⚠️ Skipping dataset '{dataset_name}': {e}")
            return None
    else:
        try:
            dataset = load_dataset(dataset_name, language_1)
        except ValueError as e:
            print(f"⚠️ Skipping language '{language_1}' for dataset '{dataset_name}': {e}")
            return None
        except Exception as e:
            print(f"❌ Unexpected error loading '{dataset_name}' ({language_1}): {e}")
            return None

    combined_data = []
    for _, split_data in dataset.items():
        combined_data.extend(split_data)

    if len(combined_data) > max_examples:
        print(f"⚠️ Dataset too large ({len(combined_data)}), sampling {max_examples} examples...")
        random.seed(42)
        combined_data = random.sample(combined_data, max_examples)

    print(f"✅ Loaded {len(combined_data)} examples")
    return combined_data


def save_splits(train: list, valid: list, lang_dir: str, subtask: str, language: str):
    """
    Save train and validation data into JSONL files.
    """
    os.makedirs(lang_dir, exist_ok=True)
    train_path = os.path.join(lang_dir, f"{subtask}_{language}.train.jsonl")
    valid_path = os.path.join(lang_dir, f"{subtask}_{language}.valid.jsonl")

    with open(train_path, "w", encoding="utf-8") as f_train:
        for ex in train:
            f_train.write(json.dumps(ex, ensure_ascii=False) + "\n")

    with open(valid_path, "w", encoding="utf-8") as f_valid:
        for ex in valid:
            f_valid.write(json.dumps(ex, ensure_ascii=False) + "\n")

    return train_path, valid_path

In [3]:

connectors={ "ht": {
    "cause": "poukisa",
    "effect": "donk sa",
},
"et": {
    "cause": "sest",
    "effect": "seetõttu",
}, 
"it":{
        "cause": "perché",
        "effect": "quindi",
    },
"id": {"cause": "karena",
        "effect": "maka"}, 

"zh": {"cause": "因为",
        "effect": "所以"}, 

"tr": { "cause": "çünkü",
        "effect": "bu yüzden"},

"eu": { "cause": "zergatik",
        "effect": "beraz"},

"en": { "cause": "because",
        "effect": "therefore"}}


In [ ]:
def get_dataset_config(dataset_name: str, language: str, subtask: str = None):
    """
    Returns the HuggingFace config name (language_1) and a column mapping
    for the given dataset_name, language, and optional subtask.
    """
    # Default
    language_1 = None
    current_columns_map = None

    # ------------------------
    # XNLI
    # ------------------------
    if dataset_name in ["facebook/xnli", "HiTZ/xnli-eu"]:
        language_1 = language  # ISO 639-1 codes
        current_columns_map = {
            "premise": "premise",
            "hypothesis": "hypothesis",
            "label": "label"
        }
    elif dataset_name == "masakhane/afrixnli":
        if language not in LANGUAGE_CODES:
            raise ValueError(f"Language '{language}' not found in FULL_LANGUAGE_NAMES")
        language_1 = ISO_639_1_TO_3[language]
        current_columns_map = {
            "premise": "premise",
            "hypothesis": "hypothesis",
            "label": "label"
        }

    # ------------------------
    # SIB200
    # ------------------------
    elif dataset_name == "Davlan/sib200":
        language_mapping = get_lang_mapping(
            yaml_path="./multilingual-evaluation/tasks/sib200/_sib200.yaml",
            language_codes=LANGUAGE_CODES
        )
        if language not in language_mapping:
            raise ValueError(f"Language '{language}' not found in SIB200 mapping")
        language_1 = language_mapping[language]
        current_columns_map = {
            "sentence": "text",
            "label": "category",
            "idx": "index_id"
        }
    
    elif dataset_name == "masakhane/afrimmlu":
        if language not in LANGUAGE_CODES:
            raise ValueError(f"Language '{language}' not found in FULL_LANGUAGE_NAMES")
        language_1 = ISO_639_1_TO_3[language]
        current_columns_map = {
            "sentence": "question",
            "label": "answer"
        }

    # ------------------------
    # MuBench (with subtasks)
    # ------------------------
    elif dataset_name == "aialt/MuBench":
        if subtask == "arc":
            language_1 = f"ARCEasyDataset_en_template_{language}"
            current_columns_map = {
            "sentence": "prompt",
            "label": "label"
        }
        elif subtask == "bmlama":
            language_1 = f"BMLAMADataset_en_template_{language}"
            current_columns_map = {
            "sentence": "prompt",
            "label": "label"
        }
        elif subtask == "mmlu":
            language_1 = f"MMLUDataset_en_template_{language}"
            current_columns_map = {
            "sentence": "prompt",
            "label": "label"
        }
        elif subtask == "truthfulqa":
            language_1 = f"TruthfulQADataset_en_template_{language}"
            current_columns_map = {
            "sentence": "prompt",
            "label": "label"
        }
        elif subtask == "mnli":
            language_1 = f"MNLIDataset_en_template_{language}"
            current_columns_map = {
            "premise": "premise",
            "hypothesis": "hypothesis",
            "label": "label"
        }
        else:
            raise ValueError(f"Unknown MuBench subtask: {subtask}")

        


    # ------------------------
    # Belebele
    # ------------------------
    elif dataset_name == "facebook/belebele":
        language_mapping = get_lang_mapping(
            yaml_path="./multilingual-evaluation/tasks/sib200/_sib200.yaml",
            language_codes=LANGUAGE_CODES
        )
        if language not in language_mapping:
            raise ValueError(f"Language '{language}' not found in Belebele mapping")
        language_1 = language_mapping[language]
        current_columns_map = {
            "sentence": "prompt",
            "label": "label"
        }

    # ------------------------
    # CohereLabs include-base-44
    # ------------------------
    elif dataset_name == "CohereLabs/include-base-44":
        if language not in FULL_LANGUAGE_NAMES:
            raise ValueError(f"Language '{language}' not found in FULL_LANGUAGE_NAMES")
        language_1 = FULL_LANGUAGE_NAMES[language]
        current_columns_map = {
            "sentence": "prompt",
            "label": "label"
        }

    # ------------------------
    # CohereLabs Global-MMLU-Lite
    # ------------------------
    
    elif dataset_name == "CohereLabs/Global-MMLU":
        if language not in LANGUAGE_CODES:
            raise ValueError(f"Language '{language}' not found in FULL_LANGUAGE_NAMES")
        language_1 = language
        current_columns_map = {
            "sentence": "question",
            "label": "answer"
        }

    # ------------------------
    # XCoPA
    # ------------------------
    elif dataset_name in ["cambridgeltl/xcopa", "pkavumba/balanced-copa", "HiTZ/XCOPA-eu"]:
        language_1 = language  # ISO 639-1 codes
        current_columns_map = {
            "sentence1": "premise",
            "sentence2": "premise",
            "label": "label"
        }

    else:
        raise ValueError(f"Dataset '{dataset_name}' not supported")

    return language_1, current_columns_map

In [11]:
import ast
def format_example(
    dataset_name: str,
    subtask: str,
    lang: str,
    ex: dict,
    idx: int,
    current_columns_map: dict,
    label_map: list,
    dataset_cleaning_regex: dict

):
    """
    Format one example into the unified record.
    Handles special cases (Belebele, Include, MuBench, etc.).
    """
    record = {"idx": idx}

    # ------------------------------
    # Scegli regex corretto per dataset/subtask
    # ------------------------------
    if dataset_name == "aialt/MuBench" and subtask in dataset_cleaning_regex:
        regex = dataset_cleaning_regex[subtask]
    elif dataset_name in dataset_cleaning_regex:
        regex = dataset_cleaning_regex[dataset_name]
    else:
        regex = dataset_cleaning_regex["default"]

    for col_name, field in current_columns_map.items():
        value = ex.get(field, "")

        # ------------------------------
        # Special cases for prompts
        # ------------------------------
        if dataset_name == "facebook/belebele" and col_name == "sentence":
            try:
                value = (
                    f"{ex.get('flores_passage', '').strip()}\n"
                    f"{ex.get('question', '').strip()}\n"
                    f"A: {ex.get('mc_answer1', '').strip()}\n"
                    f"B: {ex.get('mc_answer2', '').strip()}\n"
                    f"C: {ex.get('mc_answer3', '').strip()}\n"
                    f"D: {ex.get('mc_answer4', '').strip()}\n"
                )
            except Exception as e:
                print(f"⚠️ Error building Belebele prompt for record {idx}: {e}")
                value = ""

        elif dataset_name in ["CohereLabs/include-base-44", "CohereLabs/Global-MMLU"] and col_name == "sentence":
            try:
                value = (
                    f"{ex.get('question', '').strip()}\n"
                    f"A. {ex.get('option_a', '').strip()}\n"
                    f"B. {ex.get('option_b', '').strip()}\n"
                    f"C. {ex.get('option_c', '').strip()}\n"
                    f"D. {ex.get('option_d', '').strip()}\n"
                )
            except Exception as e:
                print(f"⚠️ Error building Include prompt for record {idx}: {e}")
                value = ""
        
        elif dataset_name == "masakhane/afrimmlu" and col_name == "sentence":
            try:
                value = (
                    f"{ex.get('question', '').strip()}\n"
                    f"A. {ast.literal_eval(ex.get('choices', ''))[0]}\n"
                    f"B. {ast.literal_eval(ex.get('choices', ''))[1]}\n"
                    f"C. {ast.literal_eval(ex.get('choices', ''))[2]}\n"
                    f"D. {ast.literal_eval(ex.get('choices', ''))[3]}\n"
                )
            except Exception as e:
                print(f"⚠️ Error building AfriMMLU prompt for record {idx}: {e}")
                value = ""

        elif dataset_name in ["cambridgeltl/xcopa", "pkavumba/balanced-copa", "HiTZ/XCOPA-eu"]:
            try:
                question = ex.get('question', '').strip()
                premise = ex.get('premise', '').strip()
                choice1 = ex.get('choice1', '').strip()
                choice2 = ex.get('choice2', '').strip()

                if col_name == 'sentence1':
                    if question == 'cause':
                        value = f"{choice1} {premise}"
                    elif question == 'effect':
                        value = f"{premise} {choice1}"
                    else:
                        value = ""

                elif col_name == 'sentence2':
                    if question == 'cause':
                        value = f"{choice2} {premise}"
                    elif question == 'effect':
                        value = f"{premise} {choice2}"
                    else:
                        value = ""

                '''question = ex.get('question', '').strip()
                premise = ex.get('premise', '').strip()
                choice1 = ex.get('choice1', '').strip()
                choice2 = ex.get('choice2', '').strip()

                if col_name == 'sentence1':
                    if question == 'cause':
                        value = f"{choice1[:-1]} {connectors[lang]['cause']} {premise.lower()}"
                    elif question == 'effect':
                        value = f"{premise[:-1]} {connectors[lang]['effect']} {choice1.lower()}"
                    else:
                        value = ""

                elif col_name == 'sentence2':
                    if question == 'cause':
                        value = f"{choice2[:-1]} {connectors[lang]['cause']} {premise.lower()}"
                    elif question == 'effect':
                        value = f"{premise[:-1]} {connectors[lang]['effect']} {choice2.lower()}"
                    else:
                        value = ""'''


            except Exception as e:
                print(f"⚠️ Error building XCoPA prompt for record {idx}: {e}")
                value = ""

    
        elif dataset_name == "aialt/MuBench" and subtask == 'mnli' and col_name == 'premise':
            try:
                # Full prompt text from HuggingFace
                full_prompt = ex.get("prompt", "").strip()

                # Extract text between "Premise:" and "Hypothesis:"
                match = re.search(r"Premise:\s*(.*?)\nHypothesis:", full_prompt, re.IGNORECASE | re.DOTALL)
                value = match.group(1).strip() if match else ""

            except Exception as e:
                print(f"⚠️ Error extracting MNLI premise for record {idx}: {e}")
                value = ""

        elif dataset_name == "aialt/MuBench" and subtask == 'mnli' and col_name == 'hypothesis':
            try:
                # Full prompt text from HuggingFace
                full_prompt = ex.get("prompt", "").strip()

                # Extract text between "Premise:" and "Hypothesis:"
                match = re.search(r"Hypothesis:\s*(.*?)\s*Question:", full_prompt, re.IGNORECASE | re.DOTALL)
                value = match.group(1).strip() if match else ""

            except Exception as e:
                print(f"⚠️ Error extracting MNLI premise for record {idx}: {e}")
                value = ""
        
        # ------------------------------
        # Special cases for labels
        # ------------------------------
        elif dataset_name == "facebook/belebele" and col_name == "label":
            correct_num = ex.get("correct_answer_num", None)
            value = label_map[int(correct_num) - 1] if correct_num else None

        elif dataset_name == "CohereLabs/include-base-44" and col_name == "label":
            correct_num = ex.get("answer", None)
            value = label_map[correct_num] if correct_num is not None else None
        
        elif dataset_name in ["CohereLabs/Global-MMLU", "masakhane/afrimmlu"] and col_name == "label":
            try:
                correct_num = ex.get("answer", None)
            except Exception as e:
                print(f"⚠️ Error mapping Include label for record {idx}: {e}")
                value = None

        elif dataset_name == "aialt/MuBench" and subtask in ["arc", "bmlama", "mmlu", "mnli","truthfulqa"] and col_name == "label":
            try:
                choices = ex.get("choices", [])
                if isinstance(choices, list) and isinstance(value, (int, str)):
                    idx_val = int(value)
                    value = choices[idx_val] if 0 <= idx_val < len(choices) else None
                else:
                    value = None
            except Exception as e:
                print(f"⚠️ Error mapping MuBench label at record {idx}: {e}")
                value = None

        # ------------------------------
        # Default: clean up strings
        # ------------------------------
        else:
            if isinstance(value, str):
                value = value.strip()
                value = regex.sub("", value).strip()

        record[col_name] = value

    return record

In [10]:
def prepare_dataset(
    dataset_name: str,
    languages: list,
    output_base_path: str,
    subtask: str = None,
    split_ratio: float = 0.2,
):
    """
    Downloads and processes a dataset from HuggingFace for specific languages.
    Saves train and validation splits as JSONL files.
    """


    label_map = ['A', 'B', 'C', 'D', 'E']  # For multiple-choice datasets like Belebele

    if not isinstance(languages, list):
        raise ValueError("`languages` must be a list of language codes.")

    os.makedirs(output_base_path, exist_ok=True)
    results = {}

    for language in languages:
        try:
            language_1, current_columns_map = get_dataset_config(dataset_name, language, subtask)
            if not language_1 or not current_columns_map:
                raise ValueError(f"No dataset config for {language}")
            
            combined_data = load_and_sample_dataset(dataset_name, language_1, max_examples=10000)
            if not combined_data:
                print(f"⚠️ No data loaded for {language}")
                continue

            formatted = [
                format_example(dataset_name, subtask, language_1, ex, idx, current_columns_map, label_map, DATASET_CLEANING_REGEX)
                for idx, ex in enumerate(combined_data)
            ]

            train, valid = train_test_split(formatted, test_size=split_ratio, random_state=42)
            lang_dir = os.path.join(output_base_path, language)
            os.makedirs(lang_dir, exist_ok=True)

            train_path, valid_path = save_splits(train, valid, lang_dir, subtask or "dataset", language)
            results[language] = {"train": train_path, "valid": valid_path}

        except ValueError as e:
            print(f"⚠️ Skipping language '{language}' for dataset '{dataset_name}': {e}")
        except Exception as e:
            print(f"❌ Unexpected error for language '{language}': {e}")

    return results


In [ ]:
# ======================
# EXAMPLE USAGE
# ======================

## to generate the train and valid split for a language or a list of language you can provide the language code(s) in the list below, 
# the Huggingface name of the dataset you are interested in and its subtask (which correspond to the config name on Huggingface)
# Here's the list among which you can choose datase_name ["cambridgeltl/xcopa","HiTZ/XCOPA-eu", "HiTZ/xnli-eu", "pkavumba/balanced-copa", "facebook/xnli", "Davlan/sib200", "aialt/MuBench", "facebook/belebele", "CohereLabs/include-base-44", "CohereLabs/Global-MMLU", "masakhane/afrimmlu", "masakhane/afrixnli"] 
# afri-mmlu and afri-nli is for all the African languages that are not included directly in the Cohere dataset or in xnli
# under the subtask you can provide "arc", "bmlama", "mnli", "truthfulqa" for MuBench and then xcopa, xnli, sib200, belebele, include, global_mmlu
base_path = "./finetuning_data "


prepare_dataset(
    dataset_name="HiTZ/xnli-eu",
    languages=["eu"],
    output_base_path=os.path.join(base_path, "xnli"),
    subtask = "xnli",
    split_ratio=0.2
)

    

⚠️ Dataset too large (400202), sampling 10000 examples...
✅ Loaded 10000 examples


{'eu': {'train': './finetuning_data_first_tier1/xnli/eu/xnli_eu.train.jsonl',
  'valid': './finetuning_data_first_tier1/xnli/eu/xnli_eu.valid.jsonl'}}